# Dynamic Pricing Simulation - Full Validation

This notebook runs a complete simulation comparing:
1. **Baseline** - Historical random pricing (what you actually did)
2. **Dynamic** - Our proposed model
3. **Low** - Always cheap (\$2.99) - maximizes orders but ignores capacity
4. **High** - Always expensive ($15.99) - maximizes revenue but hurts volume

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
np.random.seed(42)

print("✓ Libraries loaded")

✓ Libraries loaded


## Load Your Data

Update the file paths below to point to your actual data files.

In [2]:
# UPDATE THESE PATHS TO YOUR DATA FILES
# Example: '../data/raw/zipline_session_log.csv'
SESSION_FILE = '../data/raw/zipline_session_log_(5).csv'  # Update this path
PRICING_FILE = '../data/raw/zipline_pricing_schedule.csv'  # Update this path

# Load data
session_df = pd.read_csv(SESSION_FILE)
pricing_df = pd.read_csv(PRICING_FILE)

# Parse timestamps
session_df['timestamp'] = pd.to_datetime(session_df['timestamp'])
pricing_df['date_hour'] = pd.to_datetime(pricing_df['date_hour'])

print(f"✓ Loaded {len(session_df):,} sessions")
print(f"✓ Loaded {len(pricing_df):,} pricing records")
print(f"\nDate range: {session_df['timestamp'].min().date()} to {session_df['timestamp'].max().date()}")
print(f"Total days: {(session_df['timestamp'].max() - session_df['timestamp'].min()).days}")

✓ Loaded 59,880 sessions
✓ Loaded 21,495 pricing records

Date range: 2024-01-01 to 2024-01-13
Total days: 12


## Quick Data Check

In [3]:
print("Session Outcomes:")
print(session_df['session_outcome'].value_counts())
print(f"\nBlocking Rate: {(session_df['session_outcome']=='blocked').mean():.1%}")
print(f"Order Rate: {(session_df['session_outcome']=='ordered').mean():.1%}")
print(f"Abandonment Rate: {(session_df['session_outcome']=='abandoned').mean():.1%}")

print("\n" + "="*60)
print("Blocking by Hub:")
print("="*60)
blocking_by_hub = session_df.groupby('nest_id')['session_outcome'].apply(
    lambda x: (x == 'blocked').mean()
).sort_values(ascending=False)
print(blocking_by_hub)

Session Outcomes:
session_outcome
ordered      30551
abandoned    26793
blocked       2536
Name: count, dtype: int64

Blocking Rate: 4.2%
Order Rate: 51.0%
Abandonment Rate: 44.7%

Blocking by Hub:
nest_id
1     0.110326
0     0.102142
2     0.087908
3     0.082801
4     0.078990
5     0.015148
6     0.011532
7     0.008905
8     0.003415
9     0.002361
10    0.000000
11    0.000000
12    0.000000
13    0.000000
14    0.000000
Name: session_outcome, dtype: float64


## Initialize Dynamic Pricing Model

In [4]:
# Import the model
from dynamic_pricing_model import ZiplineDynamicPricing

model = ZiplineDynamicPricing()

print("✓ Model initialized")
print(f"\nHub Capacities:")
for nest_id, capacity in model.hub_capacity.items():
    print(f"  Hub {nest_id:2d}: {capacity:2d} drones")

✓ Model initialized

Hub Capacities:
  Hub  0:  5 drones
  Hub  1:  5 drones
  Hub  2:  5 drones
  Hub  3:  5 drones
  Hub  4:  5 drones
  Hub  5:  9 drones
  Hub  6:  9 drones
  Hub  7:  9 drones
  Hub  8:  9 drones
  Hub  9:  9 drones
  Hub 10: 15 drones
  Hub 11: 15 drones
  Hub 12: 15 drones
  Hub 13: 15 drones
  Hub 14: 15 drones


## Estimate Price Elasticity from Your Data

In [5]:
# Calculate conversion rates by price and order type
print("Price Elasticity Analysis")
print("="*60)

for order_type in ['Grocery', 'Meal', 'Pharma']:
    type_data = session_df[session_df['order_type'] == order_type].copy()
    
    # Create price bins
    type_data['price_bin'] = pd.cut(type_data['delivery_fee'], bins=5)
    
    # Calculate conversion by price bin
    conversion_by_price = type_data.groupby('price_bin').agg({
        'session_outcome': lambda x: (x == 'ordered').mean(),
        'delivery_fee': 'mean'
    }).round(3)
    
    conversion_by_price.columns = ['Conversion Rate', 'Avg Price']
    
    print(f"\n{order_type}:")
    print(conversion_by_price)
    
    # Calculate overall stats
    overall_conversion = (type_data['session_outcome'] == 'ordered').mean()
    median_price = type_data['delivery_fee'].median()
    
    print(f"  Overall Conversion: {overall_conversion:.1%}")
    print(f"  Median Price: ${median_price:.2f}")

Price Elasticity Analysis

Grocery:
               Conversion Rate  Avg Price
price_bin                                
(0.984, 2.19]              1.0      1.478
(2.19, 3.39]               1.0      2.990
(3.39, 4.59]               1.0      3.990
(4.59, 5.79]               1.0      4.990
(5.79, 6.99]               1.0      6.990
  Overall Conversion: 38.5%
  Median Price: $2.99

Meal:
               Conversion Rate  Avg Price
price_bin                                
(0.984, 2.19]              1.0      1.482
(2.19, 3.39]               1.0      2.990
(3.39, 4.59]               1.0      3.990
(4.59, 5.79]               1.0      4.990
(5.79, 6.99]               1.0      6.990
  Overall Conversion: 51.7%
  Median Price: $2.99

Pharma:
               Conversion Rate  Avg Price
price_bin                                
(0.984, 2.19]              1.0      1.499
(2.19, 3.39]               1.0      2.990
(3.39, 4.59]               1.0      3.990
(4.59, 5.79]               1.0      4.990
(5.79, 6

## Build Simulation Engine

In [6]:
class SimplifiedSimulator:
    """
    Simplified simulator that replays historical sessions
    under different pricing strategies.
    """
    
    def __init__(self, session_df, model):
        self.session_df = session_df.copy()
        self.model = model
        
        # Fit elasticity from data
        self.elasticity = self._fit_elasticity()
        
    def _fit_elasticity(self):
        """Estimate price elasticity by order type."""
        elasticity = {}
        
        for order_type in ['Grocery', 'Meal', 'Pharma']:
            type_data = self.session_df[self.session_df['order_type'] == order_type]
            
            elasticity[order_type] = {
                'base_conversion': (type_data['session_outcome'] == 'ordered').mean(),
                'base_price': type_data['delivery_fee'].median(),
                'blocking_rate': (type_data['session_outcome'] == 'blocked').mean()
            }
        
        return elasticity
    
    def estimate_conversion(self, price, order_type, base_price, base_conversion):
        """
        Estimate conversion rate at a given price.
        Uses simple linear elasticity:
        - Pharma: -0.5 (inelastic)
        - Meal: -1.0 (unit elastic)
        - Grocery: -1.5 (elastic)
        """
        elasticity_coef = {
            'Pharma': -0.5,
            'Meal': -1.0,
            'Grocery': -1.5
        }
        
        elasticity = elasticity_coef[order_type]
        
        # Percent change in quantity = elasticity × percent change in price
        if base_price > 0:
            price_change_pct = (price - base_price) / base_price
            conversion_change_pct = elasticity * price_change_pct
            new_conversion = base_conversion * (1 + conversion_change_pct)
        else:
            new_conversion = base_conversion
        
        # Keep conversion between 0.1 and 0.95
        return max(0.1, min(0.95, new_conversion))
    
    def simulate_hour(self, hour_sessions, strategy, current_utilization):
        """
        Simulate one hour of sessions under a pricing strategy.
        """
        results = []
        
        for _, session in hour_sessions.iterrows():
            nest_id = session['nest_id']
            order_type = session['order_type']
            distance = session['distance_miles']
            subtotal = session['subtotal']
            
            # Determine price based on strategy
            if strategy == 'baseline':
                price = session['delivery_fee']  # Historical
            elif strategy == 'dynamic':
                price = self.model.calculate_price(
                    nest_id=nest_id,
                    order_type=order_type,
                    distance_miles=distance,
                    utilization=current_utilization.get(nest_id, 0.5)
                )
            elif strategy == 'low':
                price = 2.99
            elif strategy == 'high':
                price = 15.99
            else:
                price = session['delivery_fee']
            
            # Estimate conversion probability
            base_conversion = self.elasticity[order_type]['base_conversion']
            base_price = self.elasticity[order_type]['base_price']
            
            conversion_prob = self.estimate_conversion(
                price, order_type, base_price, base_conversion
            )
            
            # Estimate blocking probability (based on utilization)
            util = current_utilization.get(nest_id, 0.5)
            if util > 0.95:
                block_prob = 0.3  # High blocking at critical capacity
            elif util > 0.85:
                block_prob = 0.15  # Moderate blocking
            elif util > 0.75:
                block_prob = 0.05  # Low blocking
            else:
                block_prob = 0.01  # Minimal blocking
            
            # Simulate outcome
            rand = np.random.random()
            
            if rand < block_prob:
                outcome = 'blocked'
                revenue = 0
            elif rand < block_prob + conversion_prob:
                outcome = 'ordered'
                revenue = price + subtotal
                # Update utilization (simplified)
                delivery_time_hrs = self.model.calculate_delivery_time(distance) / 60
                capacity = self.model.hub_capacity[nest_id]
                current_utilization[nest_id] = min(1.2, util + (delivery_time_hrs / capacity))
            else:
                outcome = 'abandoned'
                revenue = 0
            
            results.append({
                'outcome': outcome,
                'revenue': revenue,
                'price': price
            })
        
        return results
    
    def simulate_day(self, date, strategy):
        """
        Simulate one full day under a pricing strategy.
        """
        # Get sessions for this day
        day_sessions = self.session_df[
            self.session_df['timestamp'].dt.date == pd.to_datetime(date).date()
        ].copy()
        
        if len(day_sessions) == 0:
            return None
        
        # Add hour column
        day_sessions['hour'] = day_sessions['timestamp'].dt.hour
        
        # Initialize utilization tracking
        current_utilization = {i: 0.3 for i in range(15)}  # Start at 30%
        
        all_results = []
        
        # Process each hour
        for hour in range(24):
            hour_sessions = day_sessions[day_sessions['hour'] == hour]
            
            if len(hour_sessions) == 0:
                # Decay utilization during idle hours
                for nest_id in current_utilization:
                    current_utilization[nest_id] *= 0.8
                continue
            
            # Simulate this hour
            hour_results = self.simulate_hour(hour_sessions, strategy, current_utilization)
            all_results.extend(hour_results)
        
        # Calculate daily metrics
        total_sessions = len(all_results)
        total_blocks = sum(1 for r in all_results if r['outcome'] == 'blocked')
        total_orders = sum(1 for r in all_results if r['outcome'] == 'ordered')
        total_revenue = sum(r['revenue'] for r in all_results)
        
        return {
            'date': date,
            'strategy': strategy,
            'sessions': total_sessions,
            'blocks': total_blocks,
            'orders': total_orders,
            'revenue': total_revenue,
            'blocking_rate': total_blocks / total_sessions if total_sessions > 0 else 0,
            'order_rate': total_orders / total_sessions if total_sessions > 0 else 0,
            'revenue_per_session': total_revenue / total_sessions if total_sessions > 0 else 0
        }
    
    def run_comparison(self, n_days=None):
        """
        Run full comparison across all strategies.
        """
        # Get unique dates
        dates = sorted(self.session_df['timestamp'].dt.date.unique())
        
        if n_days:
            dates = dates[:n_days]
        
        strategies = ['baseline', 'dynamic', 'low', 'high']
        results = []
        
        total_simulations = len(dates) * len(strategies)
        completed = 0
        
        print(f"Running {total_simulations} simulations ({len(dates)} days × {len(strategies)} strategies)...")
        
        for date in dates:
            for strategy in strategies:
                result = self.simulate_day(str(date), strategy)
                if result:
                    results.append(result)
                
                completed += 1
                if completed % 10 == 0:
                    print(f"  Progress: {completed}/{total_simulations} ({completed/total_simulations*100:.0f}%)")
        
        print("✓ Simulation complete!")
        return pd.DataFrame(results)

print("✓ Simulator class defined")

✓ Simulator class defined


## Run Simulation

This will take a few minutes...

In [7]:
# Initialize simulator
simulator = SimplifiedSimulator(session_df, model)

# Run simulation (use None for all days, or specify n_days=7 for testing)
results_df = simulator.run_comparison(n_days=30)  # Change to None for all days

print(f"\n✓ Generated {len(results_df)} daily results")

Running 52 simulations (13 days × 4 strategies)...
  Progress: 10/52 (19%)
  Progress: 20/52 (38%)
  Progress: 30/52 (58%)
  Progress: 40/52 (77%)
  Progress: 50/52 (96%)
✓ Simulation complete!

✓ Generated 52 daily results


## Results Summary

In [8]:
# Calculate average performance by strategy
summary = results_df.groupby('strategy').agg({
    'blocking_rate': 'mean',
    'order_rate': 'mean',
    'revenue_per_session': 'mean'
}).round(4)

summary.columns = ['Avg Blocking Rate', 'Avg Order Rate', 'Avg Revenue/Session']

print("\n" + "="*80)
print("STRATEGY COMPARISON - AVERAGE PERFORMANCE")
print("="*80)
print(summary)

# Format for better readability
print("\n" + "="*80)
print("FORMATTED VIEW")
print("="*80)
for strategy in summary.index:
    print(f"\n{strategy.upper()}:")
    print(f"  Blocking Rate:      {summary.loc[strategy, 'Avg Blocking Rate']:.2%}")
    print(f"  Order Rate:         {summary.loc[strategy, 'Avg Order Rate']:.2%}")
    print(f"  Revenue per Session: ${summary.loc[strategy, 'Avg Revenue/Session']:.2f}")


STRATEGY COMPARISON - AVERAGE PERFORMANCE
          Avg Blocking Rate  Avg Order Rate  Avg Revenue/Session
strategy                                                        
baseline             0.2643          0.5930                  NaN
dynamic              0.1828          0.1375                  NaN
high                 0.1290          0.0994                  NaN
low                  0.2555          0.5003                  NaN

FORMATTED VIEW

BASELINE:
  Blocking Rate:      26.43%
  Order Rate:         59.30%
  Revenue per Session: $nan

DYNAMIC:
  Blocking Rate:      18.28%
  Order Rate:         13.75%
  Revenue per Session: $nan

HIGH:
  Blocking Rate:      12.90%
  Order Rate:         9.94%
  Revenue per Session: $nan

LOW:
  Blocking Rate:      25.55%
  Order Rate:         50.03%
  Revenue per Session: $nan


In [9]:
# Calculate improvements over baseline
baseline = summary.loc['baseline']

print("\n" + "="*80)
print("IMPROVEMENT OVER BASELINE")
print("="*80)

for strategy in ['dynamic', 'low', 'high']:
    if strategy not in summary.index:
        continue
        
    strat = summary.loc[strategy]
    
    print(f"\n{strategy.upper()} vs BASELINE:")
    
    blocking_change = (strat['Avg Blocking Rate'] / baseline['Avg Blocking Rate'] - 1) * 100
    order_change = (strat['Avg Order Rate'] / baseline['Avg Order Rate'] - 1) * 100
    revenue_change = (strat['Avg Revenue/Session'] / baseline['Avg Revenue/Session'] - 1) * 100
    
    print(f"  Blocking Rate:   {strat['Avg Blocking Rate']:.2%} vs {baseline['Avg Blocking Rate']:.2%} ({blocking_change:+.1f}%)")
    print(f"  Order Rate:      {strat['Avg Order Rate']:.2%} vs {baseline['Avg Order Rate']:.2%} ({order_change:+.1f}%)")
    print(f"  Revenue/Session: ${strat['Avg Revenue/Session']:.2f} vs ${baseline['Avg Revenue/Session']:.2f} ({revenue_change:+.1f}%)")


IMPROVEMENT OVER BASELINE

DYNAMIC vs BASELINE:
  Blocking Rate:   18.28% vs 26.43% (-30.8%)
  Order Rate:      13.75% vs 59.30% (-76.8%)
  Revenue/Session: $nan vs $nan (+nan%)

LOW vs BASELINE:
  Blocking Rate:   25.55% vs 26.43% (-3.3%)
  Order Rate:      50.03% vs 59.30% (-15.6%)
  Revenue/Session: $nan vs $nan (+nan%)

HIGH vs BASELINE:
  Blocking Rate:   12.90% vs 26.43% (-51.2%)
  Order Rate:      9.94% vs 59.30% (-83.2%)
  Revenue/Session: $nan vs $nan (+nan%)


In [ ]:
# Check if objectives are met
print("\n" + "="*80)
print("OBJECTIVE ACHIEVEMENT - DYNAMIC MODEL")
print("="*80)

if 'dynamic' in summary.index:
    dyn = summary.loc['dynamic']
    
    print("\n✓ Priority 1: Protect Customer Experience")
    blocking_reduction = (1 - dyn['Avg Blocking Rate'] / baseline['Avg Blocking Rate']) * 100
    if dyn['Avg Blocking Rate'] < baseline['Avg Blocking Rate']:
        print(f"  ✅ BLOCKING REDUCED by {blocking_reduction:.1f}%")
        print(f"     From {baseline['Avg Blocking Rate']:.2%} → {dyn['Avg Blocking Rate']:.2%}")
    else:
        print(f"  ❌ BLOCKING INCREASED (This shouldn't happen - check thresholds)")
    
    print("\n✓ Priority 2: Maximize Profit")
    revenue_increase = (dyn['Avg Revenue/Session'] / baseline['Avg Revenue/Session'] - 1) * 100
    if dyn['Avg Revenue/Session'] > baseline['Avg Revenue/Session']:
        print(f"  ✅ REVENUE INCREASED by {revenue_increase:.1f}%")
        print(f"     From ${baseline['Avg Revenue/Session']:.2f} → ${dyn['Avg Revenue/Session']:.2f}")
    else:
        print(f"  ⚠️  REVENUE DECREASED by {abs(revenue_increase):.1f}%")
        print(f"     From ${baseline['Avg Revenue/Session']:.2f} → ${dyn['Avg Revenue/Session']:.2f}")
        print(f"     (This is acceptable if blocking reduction is achieved)")
    
    print("\n✓ Priority 3: Maximize Asset Utilization")
    order_increase = (dyn['Avg Order Rate'] / baseline['Avg Order Rate'] - 1) * 100
    if dyn['Avg Order Rate'] > baseline['Avg Order Rate']:
        print(f"  ✅ ORDER RATE INCREASED by {order_increase:.1f}%")
        print(f"     From {baseline['Avg Order Rate']:.2%} → {dyn['Avg Order Rate']:.2%}")
    else:
        print(f"  ℹ️  ORDER RATE CHANGED by {order_increase:.1f}%")
        print(f"     From {baseline['Avg Order Rate']:.2%} → {dyn['Avg Order Rate']:.2%}")
else:
    print("⚠️ Dynamic strategy not found in results")

## Visualizations

In [ ]:
# Create comparison charts
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['blocking_rate', 'order_rate', 'revenue_per_session']
titles = ['Blocking Rate (Lower is Better)', 'Order Rate', 'Revenue per Session']
colors = ['red', 'green', 'blue']

for idx, (metric, title, color) in enumerate(zip(metrics, titles, colors)):
    ax = axes[idx]
    
    # Calculate means
    means = results_df.groupby('strategy')[metric].mean().sort_values()
    
    # Plot bars
    bars = ax.bar(range(len(means)), means.values, color=color, alpha=0.6, edgecolor='black')
    
    # Highlight dynamic
    if 'dynamic' in means.index:
        dyn_idx = list(means.index).index('dynamic')
        bars[dyn_idx].set_color('gold')
        bars[dyn_idx].set_edgecolor('black')
        bars[dyn_idx].set_linewidth(2)
    
    ax.set_xticks(range(len(means)))
    ax.set_xticklabels([s.capitalize() for s in means.index], rotation=0)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(means.values):
        if metric == 'revenue_per_session':
            label = f'${v:.2f}'
        else:
            label = f'{v:.1%}'
        ax.text(i, v, label, ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('/home/claude/strategy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Chart saved to strategy_comparison.png")

In [ ]:
# Time series comparison
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for idx, (metric, ylabel) in enumerate([
    ('blocking_rate', 'Blocking Rate'),
    ('order_rate', 'Order Rate'),
    ('revenue_per_session', 'Revenue per Session ($)')
]):
    ax = axes[idx]
    
    for strategy in ['baseline', 'dynamic', 'low', 'high']:
        if strategy not in results_df['strategy'].values:
            continue
            
        strategy_data = results_df[results_df['strategy'] == strategy].sort_values('date')
        
        if strategy == 'dynamic':
            ax.plot(range(len(strategy_data)), strategy_data[metric], 
                   label=strategy.capitalize(), linewidth=2.5, marker='o', markersize=4)
        else:
            ax.plot(range(len(strategy_data)), strategy_data[metric], 
                   label=strategy.capitalize(), alpha=0.5, linewidth=1.5)
    
    ax.set_ylabel(ylabel, fontsize=11)
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    if idx == 2:
        ax.set_xlabel('Day', fontsize=11)

plt.tight_layout()
plt.savefig('/home/claude/time_series_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Chart saved to time_series_comparison.png")

## Export Results

In [ ]:
# Save detailed results
results_df.to_csv('/home/claude/simulation_results.csv', index=False)

# Save summary
summary.to_csv('/home/claude/simulation_summary.csv')

print("✓ Results saved:")
print("  - simulation_results.csv (daily details)")
print("  - simulation_summary.csv (strategy averages)")
print("  - strategy_comparison.png")
print("  - time_series_comparison.png")

## Conclusion

Review the results above to see:

1. **Did dynamic pricing reduce blocking?** (Priority 1)
2. **Did it increase revenue per session?** (Priority 2)
3. **Did it improve order rates?** (Priority 3)

If all three are YES, you have a winner! 🎉

If blocking increased or revenue decreased significantly, you may need to tune the model parameters:
- Adjust capacity thresholds in `dynamic_pricing_model.py`
- Modify price ceilings
- Change elasticity coefficients

The beauty of this simple model is that it's easy to understand, explain, and tune.